# Stage 2 — SAW Recommender

**What this notebook does:** combines the two signals — *weak now* (Stage 0) and *will worsen* (Stage 1) — and uses **Simple Additive Weighting (SAW)** to rank the 5 cognitive domains for training, with transparent reasons. A separate **adapter** (built later in this notebook) routes each patient to the **AD arm**, **LBD arm**, or a **safe fallback**, based on the diagnosis probabilities from Components 1–3.

**Plan:** 1) build the criteria table → 2) SAW scoring (normalise → weight → rank → reasons) → 3) adapter + mock + safe fallback → 4) end-to-end → 5) evaluation (with/without decline, vs baseline).

**This step (Step 1):** assemble the per-patient **criteria table** — the inputs SAW will score. No scoring yet.

## Setup

Load the two signals saved by the earlier stages, and set the weakness threshold.

The **weakness threshold is a cited clinical convention** a domain is "weak" when `z ≤ −1.0 SD`. We never tune it to improve results; we will instead run a sensitivity analysis (−1.0 / −1.5 / −2.0) later.

In [ ]:
import os, json
import numpy as np, pandas as pd

PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]

# CITED clinical convention (CLAUDE_threshold_v1.md) -- DO NOT tune/auto-optimise.
# NIA-AA 1.0-1.5 SD; Heaton 1 SD; MoCA-Tamil < -1 SD.  z <= -1.0 flags a weak domain.
WEAK_THRESHOLD = -1.0

prof = pd.read_parquet(os.path.join(PROJ, "data", "cognitive_profiles.parquet"))      # weak-now: domain z-scores
decl = pd.read_parquet(os.path.join(PROJ, "data", "decline_predictions_oof.parquet"))  # will-worsen: OOF severities (leak-free)
print("profiles:", prof.shape, "| decline (out-of-fold):", decl.shape)

## Step 1 — Build the SAW criteria table

For each patient and each of the 5 domains we assemble the three inputs SAW will use:

- **weak-now severity** = `−z` — higher means more impaired *right now* (from Stage 0).
- **is_weak** = `z ≤ −1.0` — the cited weakness flag. We use it as a **criterion** (option B), so a domain predicted to *worsen into* weakness can still surface.
- **will-worsen severity** = the Stage 1 prediction — higher means faster predicted decline. We use the **out-of-fold** values so any later evaluation is leakage-free.

We keep every profiled patient. The will-worsen columns are **blank for patients without enough longitudinal data** (one visit, or no two visits that overlap on a domain) — SAW handles that in the next step. The two severities are on **different scales** (weak-now can be large; will-worsen ≈ 0–1.7), so we leave them **raw** here; SAW normalises them **within each patient** at the scoring step.

**What the weak-% table means:** it is **descriptive** — how common the flag is in each arm — **not** proof of clinical optimality or of "disease signatures" (those claims need separate clinical validation). Also note for the methodology: z-scores use the **NACC cognitively-normal participants in this dataset, without age/education adjustment**.

In [ ]:
C = prof[["NACCID", "diagnosis"]].copy()
for d in dom:
    C["weaknow_" + d] = -prof[d]                      # higher = weaker now
    C["isweak_" + d]  = prof[d] <= WEAK_THRESHOLD     # cited -1.0 SD flag (as a criterion)

C = C.merge(decl[["NACCID"] + ["severity_" + d for d in dom]], on="NACCID", how="left")
C = C.rename(columns={"severity_" + d: "willworsen_" + d for d in dom})

OUT = os.path.join(PROJ, "data", "saw_criteria.parquet")
C.to_parquet(OUT, index=False)
print("Saved:", OUT)

has_decline = C[["willworsen_" + d for d in dom]].notna().any(axis=1)
print(f"\nPatients: {len(C):,} total | with a will-worsen signal: {has_decline.sum():,}")

print("\n% of patients with each domain flagged WEAK (z <= -1.0), by arm:")
flag = C.groupby("diagnosis")[["isweak_" + d for d in dom]].mean().mul(100).round(0)
flag.columns = dom
print(flag.to_string())

print("\nPreview (memory & visuospatial columns):")
cols = ["NACCID", "diagnosis", "weaknow_memory", "isweak_memory", "willworsen_memory",
        "weaknow_visuospatial", "isweak_visuospatial", "willworsen_visuospatial"]
print(C.head(4)[cols].round(2).to_string(index=False))

## Step 2 — SAW scoring rules (defined before coding)

We rank the 5 domains per patient by a weighted sum of two criteria. Rules fixed in writing so they're defensible:

**Criteria (continuous, per domain):**
- **C1 weak-now** = `−z` (higher = more impaired now).
- **C2 will-worsen** = out-of-fold decline severity (higher = faster predicted decline).
- `is_weak` (z ≤ −1.0, cited) is used ONLY in the reason text and the sensitivity analysis — **not** as a third weighted criterion (avoids double-counting weakness).

**Normalisation:** within each patient, **min–max** each criterion across its 5 domains → 0–1. If all 5 are equal (or, for C2, fewer than 2 domains have a value), that criterion contributes 0.
*What the score means:* a **relative training priority within the patient, not absolute clinical severity** — the weakest domain scores 1 even if all domains are normal. So we also report the raw `is_weak` flags and decline values.

**Weights:** default **0.5 / 0.5** (weight-sensitivity analysis later).

**Missing decline:**
- *No decline for any domain* → **weak-now-only** ranking (output labelled).
- *Decline missing for some domains* → those domains scored on weak-now only, **weights renormalised per domain**. *Limitation:* domains scored on different criteria sit on the same 0–1 scale but are **not fully equivalent in meaning** — we record the criteria used per domain.

**Maintenance:** if a patient has **no weak domain and nothing predicted to worsen** → label the output `maintenance` (no clinical target).

**Logging:** per patient record the pathway (both / weak-now-only) and report the frequencies.

In [ ]:
import warnings
crit = pd.read_parquet(os.path.join(PROJ, "data", "saw_criteria.parquet"))

WN = crit[["weaknow_" + d for d in dom]].to_numpy(float)     # weak-now severity (-z)
WW = crit[["willworsen_" + d for d in dom]].to_numpy(float)  # will-worsen severity (NaN if no decline)
IW = crit[["isweak_" + d for d in dom]].to_numpy(bool)
w_dec = 0.3        # data-driven default from Step 5 (held-out); configurable for expert override
w_now = 1 - w_dec  # 0.7 weak-now / 0.3 will-worsen

# C1: within-patient min-max (weak-now always present)
lo, hi = WN.min(1, keepdims=True), WN.max(1, keepdims=True); rng = hi - lo
nNOW = np.where(rng > 0, (WN - lo) / np.where(rng > 0, rng, 1), 0.0)

# C2: within-patient min-max over available decline values (need >=2 to normalise)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    lo2, hi2 = np.nanmin(WW, 1, keepdims=True), np.nanmax(WW, 1, keepdims=True)
navail, rng2 = (~np.isnan(WW)).sum(1, keepdims=True), hi2 - lo2
usable = (navail >= 2) & (rng2 > 0)
nDEC = np.where(usable, (WW - lo2) / np.where(rng2 > 0, rng2, 1), np.nan)

# per-domain score with per-domain renormalised weights
has_dec = ~np.isnan(nDEC)
num = w_now * nNOW + np.where(has_dec, w_dec * np.nan_to_num(nDEC), 0.0)
den = w_now + np.where(has_dec, w_dec, 0.0)
SCORE = num / den

pathway = np.where(has_dec.any(1), "both", "weak-now-only")
maintenance = (~IW.any(1)) & (np.nan_to_num(WW) <= 0).all(1)

rec = crit[["NACCID", "diagnosis"]].copy()
for j, d in enumerate(dom):
    rec["score_" + d] = SCORE[:, j]
rec["top_domain"]  = [dom[i] for i in SCORE.argmax(1)]
rec["pathway"]     = pathway
rec["maintenance"] = maintenance
rec.to_parquet(os.path.join(PROJ, "data", "saw_recommendations.parquet"), index=False)
print("Saved: data/saw_recommendations.parquet")

print("\nPathway used:", dict(pd.Series(pathway).value_counts()))
print("Maintenance-labelled:", int(maintenance.sum()))
print("\nTop-priority domain by arm (count):")
print(rec.groupby("diagnosis")["top_domain"].value_counts().unstack().fillna(0).astype(int).to_string())

def reason(iw, ww):
    worsening = (not np.isnan(ww)) and ww > 0
    if iw and worsening: return "weak now + predicted decline -> high priority"
    if iw:               return "weak now -> priority"
    if worsening:        return "not weak now but predicted decline -> preventive"
    return "stable"

ex = int(np.where(pathway == "both")[0][0])
print(f"\n--- Example plan ({crit.iloc[ex]['NACCID']}, {crit.iloc[ex]['diagnosis']}, pathway={pathway[ex]}) ---")
for i in SCORE[ex].argsort()[::-1]:
    print(f"  {dom[i]:<13} score={SCORE[ex,i]:.2f}  | {reason(IW[ex,i], WW[ex,i])}")

## Step 3 — Integration adapter + mock + safe fallback

Components 1–3 hand us **probabilities over 4 labels** (Alzheimer, Parkinson, Vascular, LBD) — a hint, not a certainty. This cell is the **front door**: it validates that input and decides where the patient goes.

**Routing:**
- Alzheimer → **AD arm**
- LBD → **LBD arm**
- Parkinson → **LBD arm** (Parkinson's dementia folds into NACC's LBD code)
- Vascular → **safe fallback** (no vascular arm — out of scope by design, Objective IV)
- No clear leader (low confidence / mixed) → **safe fallback**

**Tolerant but strict:** accepts a dict or a 4-value list, any capitalisation, and values summing to 1 or 100 (normalised to sum 1). But it **rejects malformed input loudly** (missing/unknown label, negative, all-zero) — never a silent guess.

**Confidence gate:** route to an arm only if the top probability is **≥ 0.50** *and* leads the runner-up by **≥ 0.15**; otherwise safe fallback. (Documented routing thresholds, to be sensitivity-tested — separate from the fixed −1.0 weakness threshold.)

**Build-now principle:** a **mock** emits these probabilities in the expected shapes so the whole pipeline is testable today, before teammates finish. Every path is exercised below.

In [ ]:
# canonical label order for list-form input: [Alzheimer, Parkinson, Vascular, LBD]
LABELS  = ["alzheimer", "parkinson", "vascular", "lbd"]
ARM_MAP = {"alzheimer": "AD", "lbd": "LBD", "parkinson": "LBD", "vascular": "FALLBACK"}
CONF_MIN, MARGIN_MIN = 0.50, 0.15     # routing gate (documented, sensitivity-tested; NOT the weakness threshold)

def _parse(probs):
    if isinstance(probs, dict):
        d = {str(k).strip().lower(): v for k, v in probs.items()}
        if set(d) != set(LABELS):
            raise ValueError(f"REJECT: labels {sorted(d)} != required {LABELS}")
        vals = [d[l] for l in LABELS]
    elif isinstance(probs, (list, tuple, np.ndarray)):
        if len(probs) != 4:
            raise ValueError(f"REJECT: expected 4 values in order {LABELS}, got {len(probs)}")
        vals = list(probs)
    else:
        raise ValueError(f"REJECT: unsupported input type {type(probs).__name__}")
    v = np.array(vals, float)
    if not np.isfinite(v).all() or (v < 0).any():
        raise ValueError(f"REJECT: values must be finite and non-negative: {vals}")
    if v.sum() <= 0:
        raise ValueError("REJECT: probabilities sum to 0")
    return v / v.sum()                 # normalise (sum=1 or 100 or any positive)

def route_diagnosis(probs):
    p = _parse(probs)
    order = p.argsort()[::-1]
    top, second = LABELS[order[0]], LABELS[order[1]]
    top_p, margin = p[order[0]], p[order[0]] - p[order[1]]
    tag = f"(top {top}={top_p:.2f}, margin {margin:.2f})"
    if top_p < CONF_MIN or margin < MARGIN_MIN:
        return {"route": "FALLBACK", "arm": None, "reason": f"low confidence {tag} -> safe fallback"}
    if ARM_MAP[top] == "FALLBACK":
        return {"route": "FALLBACK", "arm": None, "reason": f"{top} out of scope {tag} -> safe fallback"}
    rr = "parkinson_proxy" if top == "parkinson" else "direct"
    note = " (provisional Lewy-body-spectrum proxy; not a PDD / diagnosis confirmation)" if top == "parkinson" else ""
    return {"route": ARM_MAP[top], "arm": ARM_MAP[top], "routing_reason": rr,
            "reason": f"{top} {tag} -> {ARM_MAP[top]} arm{note}"}

def fallback_plan():
    return {"type": "safe-fallback",
            "message": "Diagnosis input inconclusive or out of scope -> general (non-personalised) plan.",
            "plan": {d: "balanced / moderate difficulty" for d in dom}}

# ---- mock inputs covering every routing path ----
valid = {
    "clean AD":        {"Alzheimer": 0.80, "LBD": 0.10, "Parkinson": 0.05, "Vascular": 0.05},
    "clean LBD":       {"Alzheimer": 0.10, "LBD": 0.78, "Parkinson": 0.07, "Vascular": 0.05},
    "Parkinson->LBD":  {"Alzheimer": 0.12, "LBD": 0.18, "Parkinson": 0.65, "Vascular": 0.05},
    "Vascular->FB":    {"Alzheimer": 0.15, "LBD": 0.10, "Parkinson": 0.05, "Vascular": 0.70},
    "mixed(margin)->FB": {"Alzheimer": 0.50, "LBD": 0.42, "Parkinson": 0.05, "Vascular": 0.03},
    "low-conf->FB":    {"Alzheimer": 0.45, "LBD": 0.25, "Parkinson": 0.20, "Vascular": 0.10},
    "list form (AD)":  [0.82, 0.06, 0.04, 0.08],
    "percent sum=100": {"Alzheimer": 70, "LBD": 15, "Parkinson": 10, "Vascular": 5},
}
print("ROUTING TEST (valid inputs):")
for name, m in valid.items():
    r = route_diagnosis(m)
    print(f"  {name:<18} -> {r['route']:<9} | {r['reason']}")

print("\nMALFORMED INPUTS (must be rejected loudly):")
bad = {
    "missing a label": {"Alzheimer": 0.5, "LBD": 0.3, "Parkinson": 0.2},
    "unknown label":   {"Alzheimer": 0.5, "LBD": 0.3, "Parkinson": 0.1, "Dementia": 0.1},
    "negative value":  {"Alzheimer": -0.2, "LBD": 0.6, "Parkinson": 0.3, "Vascular": 0.3},
    "all zeros":       {"Alzheimer": 0, "LBD": 0, "Parkinson": 0, "Vascular": 0},
}
for name, m in bad.items():
    try:
        route_diagnosis(m); print(f"  {name:<18} -> NOT rejected (BUG!)")
    except ValueError as e:
        print(f"  {name:<18} -> {e}")

print("\nSafe fallback plan:", fallback_plan()["message"])

## Step 4 — End-to-end: mock → adapter → arm → SAW → plan

This ties everything together. For a patient we feed diagnosis probabilities through the adapter; if it routes to an arm we build the SAW plan from that patient's weak-now + will-worsen; if it routes to fallback we return the safe default. We run one real patient through each path.

*Deployment note:* here the mock probabilities agree with each patient's known NACC arm, so the routed arm matches the arm whose decline model produced their will-worsen severity. In the live system the adapter's routed arm selects the decline model at run time.

In [ ]:
def recommend(nacc_id, probs, w_now=0.7, w_dec=0.3):   # 0.3 decline default (Step 5); override per call
    r = route_diagnosis(probs)
    if r["route"] == "FALLBACK":
        return {"patient": nacc_id, "route": "FALLBACK", "reason": r["reason"], **fallback_plan()}
    row = crit.loc[crit["NACCID"] == nacc_id]
    if row.empty:
        raise ValueError(f"patient {nacc_id} not in criteria table")
    row = row.iloc[0]
    wn = np.array([row["weaknow_" + d] for d in dom], float)
    ww = np.array([row["willworsen_" + d] for d in dom], float)
    iw = np.array([row["isweak_" + d] for d in dom], bool)

    rng = wn.max() - wn.min()
    nNOW = (wn - wn.min()) / rng if rng > 0 else np.zeros(5)      # C1 within-patient min-max
    hasd = ~np.isnan(ww); nDEC = np.full(5, np.nan)              # C2 over available, need >=2
    if hasd.sum() >= 2 and np.nanmax(ww[hasd]) != np.nanmin(ww[hasd]):
        lo, hi = np.nanmin(ww[hasd]), np.nanmax(ww[hasd])
        nDEC[hasd] = (ww[hasd] - lo) / (hi - lo)
    hd = ~np.isnan(nDEC)
    score = (w_now * nNOW + np.where(hd, w_dec * np.nan_to_num(nDEC), 0.0)) / (w_now + np.where(hd, w_dec, 0.0))

    order = score.argsort()[::-1]
    plan = [{"domain": dom[i], "score": round(float(score[i]), 2), "weak_now": bool(iw[i]),
             "will_worsen": (None if np.isnan(ww[i]) else round(float(ww[i]), 2)),
             "reason": reason(iw[i], ww[i])} for i in order]
    return {"patient": nacc_id, "route": r["route"], "arm": r["arm"], "reason": r["reason"],
            "pathway": ("both" if hd.any() else "weak-now-only"), "plan": plan}

def show(rec):
    if rec["route"] == "FALLBACK":
        print(f"[{rec['patient']}] route=FALLBACK :: {rec['reason']}")
        print(f"    -> {rec['message']}")
        return
    print(f"[{rec['patient']}] route={rec['route']} ({rec['reason']}) | pathway={rec['pathway']}")
    print("    (score 0-1 = relative priority within patient; decline_sev = relative decline signal, NOT a probability)")
    for p in rec["plan"]:
        ww = "n/a" if p["will_worsen"] is None else p["will_worsen"]
        print(f"    {p['domain']:<13} score={p['score']:.2f}  weak_now={str(p['weak_now']):<5} decline_sev={str(ww):<5} | {p['reason']}")

# diagnosis-probability inputs (mock Components 1-3)
P_AD   = {"Alzheimer": 0.80, "LBD": 0.10, "Parkinson": 0.05, "Vascular": 0.05}
P_LBD  = {"Alzheimer": 0.10, "LBD": 0.78, "Parkinson": 0.07, "Vascular": 0.05}
P_PARK = {"Alzheimer": 0.12, "LBD": 0.18, "Parkinson": 0.65, "Vascular": 0.05}
P_VASC = {"Alzheimer": 0.15, "LBD": 0.10, "Parkinson": 0.05, "Vascular": 0.70}
P_MIX  = {"Alzheimer": 0.50, "LBD": 0.42, "Parkinson": 0.05, "Vascular": 0.03}

# pick real patients for each pathway
ddom = ["willworsen_" + d for d in dom]
ad_both = crit[(crit.diagnosis == "AD")  & crit[ddom].notna().any(axis=1)]["NACCID"].iloc[0]
lbd_both = crit[(crit.diagnosis == "LBD") & crit[ddom].notna().any(axis=1)]["NACCID"].iloc[0]
ad_only = crit[(crit.diagnosis == "AD")  & crit[ddom].isna().all(axis=1)]["NACCID"].iloc[0]

print("=== 1) AD patient, AD-dominant input -> AD arm, full plan ===");            show(recommend(ad_both, P_AD))
print("\n=== 2) LBD patient, LBD-dominant input -> LBD arm ===");                 show(recommend(lbd_both, P_LBD))
print("\n=== 3) LBD patient, PARKINSON-dominant input -> LBD arm (PDD folds in) ==="); show(recommend(lbd_both, P_PARK))
print("\n=== 4) AD patient with no decline data -> weak-now-only plan ===");      show(recommend(ad_only, P_AD))
print("\n=== 5) any patient, VASCULAR-dominant -> safe fallback ===");            show(recommend(ad_both, P_VASC))
print("\n=== 6) any patient, MIXED / low-margin -> safe fallback ===");           show(recommend(ad_both, P_MIX))

## Step 4b — Software tests for the SAW calculation

Claim A — *"for this patient the recommender ranked X first"* — is only trustworthy if the calculation is implemented correctly. These unit tests check the SAW scoring on known inputs and edge cases (all-equal weak-now; decline missing for some / one / no domains). If any `assert` fails, the pipeline is wrong and must be fixed before any result is reported.

In [ ]:
import numpy as np
def saw_score(wn, ww, w_now=0.5, w_dec=0.5):
    """Canonical SAW per-domain score (mirrors Step 2 / Step 4). Arrays in, scores out."""
    wn = np.asarray(wn, float); ww = np.asarray(ww, float)
    rng = wn.max() - wn.min()
    nNOW = (wn - wn.min()) / rng if rng > 0 else np.zeros(len(wn))
    hasd = ~np.isnan(ww); nDEC = np.full(len(wn), np.nan)
    if hasd.sum() >= 2 and np.nanmax(ww[hasd]) != np.nanmin(ww[hasd]):
        lo, hi = np.nanmin(ww[hasd]), np.nanmax(ww[hasd]); nDEC[hasd] = (ww[hasd] - lo) / (hi - lo)
    hd = ~np.isnan(nDEC)
    return (w_now * nNOW + np.where(hd, w_dec * np.nan_to_num(nDEC), 0.0)) / (w_now + np.where(hd, w_dec, 0.0))

nan = np.nan
s = saw_score([0,1,2,3,4], [0,0,0,0,4]);        assert s.argmax()==4 and np.isclose(s[4],1.0)       # both criteria
s = saw_score([2,2,2,2,2], [0,1,2,3,4]);        assert np.allclose(s, 0.5*np.array([0,.25,.5,.75,1]))# equal weak-now -> C1=0
s = saw_score([0,1,2,3,4], [nan,nan,nan,0,2]);  assert np.isclose(s[0],0.0) and np.isclose(s[4],1.0) # per-domain renorm
s = saw_score([0,1,2,3,4], [nan]*5);            assert np.allclose(s, np.array([0,1,2,3,4])/4)       # no decline -> weak-now-only
s = saw_score([0,1,2,3,4], [nan,nan,nan,nan,3]);assert np.allclose(s, np.array([0,1,2,3,4])/4)       # single value -> can't normalise
s = saw_score([1,1,1,1,1], [nan]*5);            assert np.allclose(s, 0)                             # no differentiation
print("All SAW unit tests passed.")

## Step 4b — Cohort-level face validity (descriptive, not proof)

A single patient cannot establish a clinical pattern. Here we look across the **whole cohort** at how often each domain is ranked top, per arm.

This is descriptive face validity: is the output *plausibly consistent* with the literature — AD → memory; DLB → visuospatial / attention / executive, with memory relatively spared? that needs an expert gold-standard rubric and external validation, and it reflects the fixed −1.0 weakness threshold and the simplified (no age/education) norming.

In [ ]:
import os
import pandas as pd
try:
    PROJ
except NameError:
    PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]

rec = pd.read_parquet(os.path.join(PROJ, "data", "saw_recommendations.parquet"))
tab = (rec.groupby("diagnosis")["top_domain"].value_counts(normalize=True).mul(100).round(0)
          .unstack().fillna(0).astype(int))
tab = tab[[d for d in dom if d in tab.columns]]
print("Top-ranked domain per arm (% of patients) ")
print(tab.to_string())

## Step 5 — Evaluation + weight selection (patient-level, held-out)

We must not pick the weight on the same patients we score it on (selection optimism), so we use nested patient-level cross-validation.

- **Observed ideal** (the yardstick) = `SAW(weak-now + OBSERVED decline severity, 0.5/0.5)` per patient — the ranking implied by what actually happened (current weakness + real later decline). *This is a ranking-correspondence proxy; it does **not** prove that training those domains improves cognition — that needs an intervention study (future work).*
- Split patients into 5 folds (all of a patient's data stays together).
- **Select** the decline weight (candidates 0.0–0.5) on **training** patients only.
- **Evaluate** that weight on the **held-out** patients; repeat; summarise.
- Report held-out agreement for **weakness-only (0.0)**, **equal (0.5)** and the **nested-selected** weight; the gain + bootstrap CI; which weight each fold picked; AD/LBD separately; N included/excluded.

*Leakage note:* Stage 1 decline predictions are out-of-fold (leak-free for the decline model); the outer split here ensures the **weight** isn't chosen on the evaluation patients. Stage 1 is **not** re-fit inside the outer folds (documented limitation). Objective weights (CRITIC/entropy) and expert weights (BWM) are planned supplementary comparisons.

**Validity audit (passed, with one documented caveat).** Patients are independent (one row each → one fold); the ideal uses **observed** outcomes only (not the predicted signal, not the evaluated rankings); the weight is selected on **training** patients and scored on **held-out**; within-patient min–max has **no cross-patient parameters** to leak; CIs **resample patients**, not domain rows.

*Caveat (provenance):* Stage 1's out-of-fold predictions are leak-free at model-fitting, but Stage 1's **design choices** (feature set, the ±4 clip, the hybrid group-average fallback for LBD memory/executive) were decided on the **full** dataset, not re-derived inside each outer fold — a documented prototype limitation.

*Population:* results apply to the **5,447** patients with ≥3 domains having **both** predicted and observed decline; patients without decline data use the weak-now-only path (weight irrelevant). So **0.3 is a provisional, data-driven default**, kept configurable for expert override.

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import KFold
try:
    PROJ
except NameError:
    PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]
CANDS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]           # candidate decline weights

def saw(wn, ww, w_dec):
    w_now = 1 - w_dec
    wn = np.asarray(wn, float); ww = np.asarray(ww, float)
    rng = wn.max() - wn.min()
    nNOW = (wn - wn.min()) / rng if rng > 0 else np.zeros(len(wn))
    hasd = ~np.isnan(ww); nDEC = np.full(len(wn), np.nan)
    if hasd.sum() >= 2 and np.nanmax(ww[hasd]) != np.nanmin(ww[hasd]):
        lo, hi = np.nanmin(ww[hasd]), np.nanmax(ww[hasd]); nDEC[hasd] = (ww[hasd] - lo) / (hi - lo)
    hd = ~np.isnan(nDEC)
    return (w_now * nNOW + np.where(hd, w_dec * np.nan_to_num(nDEC), 0.0)) / (w_now + np.where(hd, w_dec, 0.0))

crit = pd.read_parquet(os.path.join(PROJ, "data", "saw_criteria.parquet"))
tab  = pd.read_parquet(os.path.join(PROJ, "data", "decline_model_table.parquet"))
for d in dom:
    tab["obs_" + d] = np.clip(-tab["decline_" + d], 0, None)      # observed decline severity
m = crit.merge(tab[["NACCID"] + ["obs_" + d for d in dom]], on="NACCID", how="inner")

rows = []
for _, r in m.iterrows():
    wn = np.array([r["weaknow_" + d] for d in dom], float)
    pr = np.array([r["willworsen_" + d] for d in dom], float)
    ob = np.array([r["obs_" + d] for d in dom], float)
    av = (~np.isnan(pr)) & (~np.isnan(ob))
    if av.sum() >= 3:
        rows.append((wn[av], pr[av], ob[av], r["diagnosis"]))
n0 = len(rows)

# per-patient agreement with the observed ideal at each candidate weight
C, arms = [], []
for wn, pr, ob, arm in rows:
    ideal = saw(wn, ob, 0.5)
    C.append([spearmanr(saw(wn, pr, w), ideal).correlation for w in CANDS]); arms.append(arm)
C = np.array(C); arms = np.array(arms)
valid = ~np.isnan(C).any(1); C, arms = C[valid], arms[valid]; n = len(C)
print(f"patients: {n0} with >=3 both-signal domains; {n} rankable; excluded {n0 - n}\n")

# nested CV: choose weight on TRAIN fold, score on HELD-OUT fold
held_sel = np.full(n, np.nan); sel_per_fold = []
for tr, te in KFold(5, shuffle=True, random_state=0).split(np.arange(n)):
    j = C[tr].mean(0).argmax(); sel_per_fold.append(CANDS[j]); held_sel[te] = C[te, j]
held_w0, held_w5 = C[:, CANDS.index(0.0)], C[:, CANDS.index(0.5)]

print("Held-out agreement with observed ideal (mean Spearman):")
print(f"   weakness-only (0.0): {held_w0.mean():.3f}")
print(f"   equal        (0.5): {held_w5.mean():.3f}")
print(f"   nested-selected    : {held_sel.mean():.3f}")
print(f"weight selected per fold: {sel_per_fold}")

rng = np.random.default_rng(0)
print("\nGain of nested-selected vs weakness-only (held-out):")
for lab, mask in [("ALL", np.ones(n, bool)), ("AD", arms == "AD"), ("LBD", arms == "LBD")]:
    g = held_sel[mask] - held_w0[mask]
    boot = np.array([g[rng.integers(0, len(g), len(g))].mean() for _ in range(2000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"   {lab:<4} n={mask.sum():5d}  gain={g.mean():+.3f}  95%CI[{lo:+.3f},{hi:+.3f}]  {'RELIABLE' if lo > 0 else 'ns'}")

## Step 6 — Threshold sensitivity (−1.0 / −1.5 / −2.0 SD)

The weakness threshold is a fixed, cited convention (−1.0 SD); the governance doc requires we **test** it, not tune it. We report how the output changes at −1.0 / −1.5 / −2.0.

**Key design point:** the SAW score uses the **continuous** weak-now severity (`−z`), **not** the threshold. So the **ranked training targets are threshold-independent** — the cutoff only changes which domains are *labelled* "weak" (reason text) and the maintenance count. 

In [ ]:
import os, numpy as np, pandas as pd
try:
    PROJ
except NameError:
    PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]

crit = pd.read_parquet(os.path.join(PROJ, "data", "saw_criteria.parquet"))
worsen_any = (crit[["willworsen_" + d for d in dom]] > 0).any(axis=1)

print("Threshold sensitivity (weakness cutoff z <= t):\n")
print(f"{'threshold':>10} | {'>=1 weak':>9} | {'mean #weak':>10} | {'maintenance':>11}")
for t in [-1.0, -1.5, -2.0]:
    W = pd.DataFrame({d: crit["weaknow_" + d] >= -t for d in dom})   # z<=t  <=>  -z >= -t
    anyweak = W.any(axis=1); maint = (~anyweak) & (~worsen_any)
    print(f"{('z<=%.1f' % t):>10} | {anyweak.mean()*100:8.0f}% | {W.sum(axis=1).mean():10.2f} | {int(maint.sum()):11d}")

print("\nBy arm (% of patients with >=1 weak domain):")
for t in [-1.0, -1.5, -2.0]:
    anyweak = pd.DataFrame({d: crit["weaknow_" + d] >= -t for d in dom}).any(axis=1)
    print(f"   z<={t}: {crit.assign(a=anyweak).groupby('diagnosis')['a'].mean().mul(100).round(0).to_dict()}")

print("\nRanking is threshold-INDEPENDENT: SAW scores the continuous -z, not the flag.")
print("-> ranked training targets are identical across thresholds; only the 'weak' labels + maintenance change.")

## Step 7 — Map ranked domains to exercises

The SAW ranking gives the **priority domains**; this step turns them into an actual **exercise plan** using the project's evidence-based catalog (`exercise_catalog.json`; Hill 2017 effect sizes).

**Selection rule (domain ranking unchanged):** for each top-ranked domain, pick the catalog exercise that (a) matches the patient's **arm** (`arm_emphasis`) and (b) has the higher **trainability** (`expected_benefit_g`); **flag low-benefit** targets (g < 0.30) rather than hide them.

**Honest flag it surfaces:** the LBD-priority domains (visuospatial g=0.18, executive g=0.20) are the *weakest* training responders, so an LBD plan often tops out on a LOW-benefit exercise. The system **shows** this (the catalog's trainability dial) so a clinician can temper expectations or lean on compensatory strategies / strong responders (working-memory n-back g=0.74).

**Governance:** the catalog is a **DRAFT** — exercises are cited paradigms, but the domain/arm/theme tagging needs **clinical sign-off** (proposal Obj I–II). Difficulty (adaptive staircase) and theme (personalisation) are **separate dials**, not set here. Trainability as a ranking criterion is a documented future option (not a change to the validated weights).

In [ ]:
import os, json, numpy as np, pandas as pd
try:
    PROJ
except NameError:
    PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]

def saw(wn, ww, w_dec=0.3):
    w_now = 1 - w_dec
    wn = np.asarray(wn, float); ww = np.asarray(ww, float)
    rng = wn.max() - wn.min()
    nNOW = (wn - wn.min()) / rng if rng > 0 else np.zeros(len(wn))
    hasd = ~np.isnan(ww); nDEC = np.full(len(wn), np.nan)
    if hasd.sum() >= 2 and np.nanmax(ww[hasd]) != np.nanmin(ww[hasd]):
        lo, hi = np.nanmin(ww[hasd]), np.nanmax(ww[hasd]); nDEC[hasd] = (ww[hasd] - lo) / (hi - lo)
    hd = ~np.isnan(nDEC)
    return (w_now * nNOW + np.where(hd, w_dec * np.nan_to_num(nDEC), 0.0)) / (w_now + np.where(hd, w_dec, 0.0))

crit = pd.read_parquet(os.path.join(PROJ, "data", "saw_criteria.parquet"))
EX = json.load(open(os.path.join(PROJ, "exercise_catalog.json"), encoding="utf-8"))["exercises"]

def plan_for(nacc_id, top_k=3, w_dec=0.3):
    row = crit.loc[crit.NACCID == nacc_id].iloc[0]; arm = row["diagnosis"]
    wn = np.array([row["weaknow_" + d] for d in dom]); ww = np.array([row["willworsen_" + d] for d in dom])
    score = saw(wn, ww, w_dec); order = np.argsort(score)[::-1]
    print(f"{nacc_id} ({arm}) -- exercise plan (top {top_k} domains, decline weight {w_dec}):")
    for i in order[:top_k]:
        d = dom[i]
        e = sorted([x for x in EX if x["domain"] == d],
                   key=lambda x: (arm in x["arm_emphasis"], x["expected_benefit_g"]), reverse=True)[0]
        fit = "arm-matched" if arm in e["arm_emphasis"] else "general"
        flag = "  [LOW-benefit]" if e["expected_benefit_g"] < 0.30 else ""
        print(f"   {d:<13} score={score[i]:.2f} -> {e['id']} {e['name']} "
              f"[{e['strategy']}, g={e['expected_benefit_g']}, {fit}, lang={e['language_dependency']}]{flag}")

ad  = crit[(crit.diagnosis == "AD")  & crit[["willworsen_" + d for d in dom]].notna().any(axis=1)]["NACCID"].iloc[0]
lbd = crit[(crit.diagnosis == "LBD") & crit[["willworsen_" + d for d in dom]].notna().any(axis=1)]["NACCID"].iloc[0]
plan_for(ad); print(); plan_for(lbd)